# Using the PlayerData API with Direct

This notebook demonstrates how to use the PlayerData API with Direct Queries to query session details and raw data.

## Prerequisites

Before using this notebook, you will need to:
- Create a virtual environment
- Install all dependencies from the `pyproject.toml` file

## Authentication

First, we need to authenticate with the PlayerData API. There are three authentication methods available:

1. **Client Credentials Flow** (shown in this example)
2. **Authorization Code Flow**
3. **Authorization Code Flow with PKCE**

To use a different authentication method, change the `authentication_type` parameter to `AuthenticationType.AUTHORIZATION_CODE_FLOW` or `AuthenticationType.AUTHORIZATION_CODE_FLOW_WITH_PKCE` and provide the required parameters.

In [ ]:
from playerdatapy.gqlauth import GraphqlAuth
from playerdatapy.gqlauth import AuthenticationType
from playerdatapy.gqlclient import Client
from playerdatapy.constants import API_BASE_URL
import httpx

# Add your client id, client secret and one of your club ids
CLIENT_ID = "your_client_id"
CLIENT_SECRET = "your_client_secret"
CLUB_ID = "your_club_id"

# Example usage of the GraphqlClient class.
auth = GraphqlAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    type=AuthenticationType.CLIENT_CREDENTIALS_FLOW,
)

# Create a Client instance.
client = Client(
    url=f"{API_BASE_URL}/api/graphql",
    http_client=httpx.AsyncClient(auth=auth.httpx_auth),
)

## Defining Queries

There are example queries in the `queries` folder that you can use as a starting point. For example, `ClubSessionsFilteredByTimeRange` retrieves all sessions for a club in a specified time period.

## Running Example Queries

For now, let's use the example queries, run them, and print the results.

In [ ]:
from datetime import datetime, timedelta

with open("queries/club_sessions_filtered_by_time_range.graphql", "r") as f:
    club_sessions_filtered_by_time_range = f.read()

# Query for all sessions in the last 30 days
start_time = datetime.now() - timedelta(days=30)
end_time = datetime.now()

last_thirty_days_sessions_response = await client.execute(
    query=club_sessions_filtered_by_time_range,
    variables={"clubId": CLUB_ID, "startTime": start_time, "endTime": end_time},
)

last_thirty_days_sessions_response = client.get_data(last_thirty_days_sessions_response)

print(last_thirty_days_sessions_response)

## Getting Session Details

From the response above, we can see the sessions returned. Now let's get detailed information for a specific session using the session ID from the first session in the response.

In [ ]:
with open("queries/session_details.graphql", "r") as f:
    session_details = f.read()

# Extract the session ID from the first session
first_session_id = last_thirty_days_sessions_response["sessions"][0]["id"]

session_details_response = await client.execute(
    query=session_details, variables={"sessionId": first_session_id}
)

session_details_response = client.get_data(session_details_response)
print(session_details_response)

## Getting Session Metrics

For the same session, you can retrieve configured metrics at different levels:
- Team aggregate level
- Session participation level
- Segment level

In [ ]:
with open("queries/session_metrics.graphql", "r") as f:
    session_metrics = f.read()

# Query for session metrics at all levels
session_metrics_response = await client.execute(
    query=session_metrics, variables={"sessionId": first_session_id}
)

session_metrics_response = client.get_data(session_metrics_response)
print(session_metrics_response)

## Raw Data Exports

Request the first session participation's GPS data as CSV with the `requestRawDataExport` mutation. The export is prepared in the background, so call the mutation again until `status` is `READY`, then download the file from `downloadUrl` (valid for 15 minutes).

See the [raw data exports guide](https://playerdata.github.io/playerdatapy/raw-data/) for other data types, whole-session exports and the data schemas.

In [ ]:
import asyncio
import io

import httpx
import polars as pl

with open("queries/request_raw_data_export.graphql", "r") as f:
    request_raw_data_export = f.read()

first_session_participation_id = session_details_response["session"][
    "sessionParticipations"
][0]["id"]

while True:
    response = await client.execute(
        query=request_raw_data_export,
        variables={
            "id": first_session_participation_id,
            "dataType": "GPS",
            "format": "CSV",
        },
    )
    result = client.get_data(response)["requestRawDataExport"]
    if result["status"] != "PROCESSING":
        break
    await asyncio.sleep(15)

print(result["status"])
if result["status"] == "READY":
    async with httpx.AsyncClient(timeout=60) as http:
        download = await http.get(result["downloadUrl"])
        download.raise_for_status()
    gps = pl.read_csv(io.BytesIO(download.content))
    print(gps.head())